# Detección + tracking de movimiento desde dron (v5): comparativa de trackers

Compara tres formas de seguir objetos, **con las mismas detecciones y la misma decisión de movimiento**, para que la única diferencia sea el tracker:

| Variante | Qué es |
|---|---|
| `propio` | Nuestro tracker: asociación estilo ByteTrack con las cajas ya compensadas por la homografía |
| `bytetrack` | ByteTrack real de Ultralytics (Kalman en coordenadas de imagen, sin compensar la cámara) |
| `botsort` | BoT-SORT real de Ultralytics con compensación global de movimiento (GMC, `sparseOptFlow`), sin ReID |
| `botsort_sin_gmc` *(opcional)* | BoT-SORT con GMC desactivado, para aislar cuánto aporta la compensación |

**Cómo se hace justa la comparación:** la homografía y las detecciones (YOLO + propuestas de movimiento) se calculan **una vez por frame** y se reparten a todos. Los tracks externos se convierten (`TrackBook`) a nuestros objetos `Track` y pasan por el mismo `MotionState`, filtro de paralaje y dibujo.

**Lo que hay que saber**
- ByteTrack y BoT-SORT son *algoritmos*, no modelos entrenados. Solo BoT-SORT puede usar un modelo ReID (aquí desactivado: no hace falta, y exigiría pesos adicionales).
- Las cajas que devuelven los trackers de Ultralytics salen del **filtro de Kalman** (suavizadas), no de la detección cruda. Es una diferencia legítima de diseño; afecta la medida de desplazamiento.
- El GMC de Ultralytics está envuelto en un `try/except` silencioso: si falla, BoT-SORT **no compensaría** la cámara sin avisar. Por eso existe `selftest_ultralytics()`: **ejecútala primero**.
- Las métricas no usan verdad de terreno; son proxies (ver abajo). El juicio final es visual con `mosaico.mp4`.

In [1]:
import csv
import os
from collections import deque, defaultdict
from dataclasses import dataclass

import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment

## 0. CONFIGURACIÓN (todas las distancias en píxeles de la resolución de trabajo)

In [2]:
@dataclass
class Config:
    work_width: int = 960          # ancho al que se estima el movimiento global y se hace el tracking
    out_width: int = 1920          # ancho del video de salida (0 = resolución original)
    # --- movimiento global (cámara)
    min_inliers: int = 30          # por debajo, la homografía se considera poco fiable
    ransac_px: float = 1.5         # tolerancia RANSAC: más baja = se ajusta más al plano del suelo
    grid: tuple = (4, 3)           # (cols, filas): reparte los puntos por toda la imagen (evita que domine una zona)
    # --- asociación (tracker)
    conf_high: float = 0.35        # detecciones "fuertes" (1ª asociación, crean tracks nuevos)
    conf_low: float = 0.10         # detecciones "débiles" (2ª asociación, estilo ByteTrack)
    match_iou: float = 0.20        # IoU mínimo (con cajas infladas) para asociar
    box_pad: float = 3.0           # px con que se inflan las cajas para el IoU (objetos pequeños)
    max_miss_sec: float = 1.0      # segundos que un track sobrevive sin detección
    min_hits: int = 4              # detecciones para confirmar un track
    # --- decisión movimiento / estático
    window_sec: float = 1.0        # ventana temporal para medir el desplazamiento neto
    min_move_px: float = 6.0       # desplazamiento mínimo (en la ventana) para considerar movimiento
    rel_move: float = 0.30         # ...y además >= rel_move * tamaño del objeto (sqrt(ancho*alto)); frena cajas grandes inestables
    adaptive: bool = False         # umbral adaptativo con la población de tracks (solo si la mayoría es estática)
    max_adaptive_px: float = 12.0  # tope del umbral adaptativo
    mad_k: float = 4.0             # umbral adaptativo = mediana + k*MAD de la población de tracks
    max_box_area_frac: float = 0.03  # se descartan detecciones con caja mayor a esta fracción del frame
    on_frames: int = 3             # evaluaciones seguidas "moviéndose" para activar el estado
    off_frames: int = 10           # evaluaciones seguidas "quieto" para desactivarlo
    # --- propuestas de movimiento agnósticas a la clase (objetos que YOLO no conoce)
    use_proposals: bool = True
    proposal_gap_sec: float = 0.15 # separación temporal entre los frames que se restan
    diff_thr: int = 25
    min_area: int = 40
    max_area: int = 6000
    dilate: int = 5
    border: int = 15
    max_aspect: float = 5.0
    max_residual: float = 3.5      # diferencia media tras alinear; si se supera, se omiten propuestas
    proposal_conf: float = 0.5
    max_coherence: float = 0.80    # estructura de una sola orientación (cables, líneas, bordillos) -> no es un objeto
    min_straightness: float = 0.60 # desplazamiento neto / recorrido en la ventana; un objeto real avanza en línea, el ruido deambula
    # --- asignación de clase (evita que una detección suelta de baja confianza cambie la etiqueta de un track)
    cls_min_conf: float = 0.40     # solo las detecciones con esta confianza votan por una clase
    cls_min_hits: int = 3          # votos mínimos para que el track adopte una clase; si no, queda "desconocido"
    # --- filtro de paralaje (lo que sobresale del suelo se desplaza respecto a él al moverse el dron)
    parallax_filter: bool = True
    parallax_ratio: float = 0.15   # h/Z máximo plausible (altura del objeto / distancia a la cámara)
    parallax_angle_deg: float = 30.0   # el residuo debe ser paralelo al flujo del suelo en ese punto (mismo sentido)
    parallax_skip: tuple = ("car", "truck", "bus", "motorcycle", "small vehicle", "large vehicle")  # clases que NO se filtran
    show_parallax: bool = True     # dibujar en gris lo descartado por paralaje
    debug_residual: bool = False   # guarda un segundo video con el residuo (frame - frame previo compensado) y las cajas
    # --- fusión de fragmentos que se mueven juntos (un vehículo largo = una sola caja)
    merge_motion: bool = True
    merge_gap: float = 0.6         # separación máxima entre cajas, como fracción del lado mayor
    merge_angle_deg: float = 25.0  # diferencia máxima de dirección
    merge_speed_ratio: float = 0.35  # diferencia relativa máxima de velocidad

## 1. MOVIMIENTO GLOBAL DE LA CÁMARA

In [3]:
def warp_pts(H, pts):
    pts = np.asarray(pts, np.float32).reshape(-1, 2)
    if len(pts) == 0:
        return pts
    return cv2.perspectiveTransform(pts.reshape(-1, 1, 2), H).reshape(-1, 2)


class GlobalMotion:
    """Estima la homografía H (frame previo -> frame actual) con LK + RANSAC."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.prev = None

    def _features(self, gray):
        """Puntos repartidos en una rejilla, para que ninguna zona (techos, pastizal) domine la homografía."""
        h, w = gray.shape
        gx, gy = self.cfg.grid
        pts = []
        for i in range(gy):
            for j in range(gx):
                y0, y1, x0, x1 = i * h // gy, (i + 1) * h // gy, j * w // gx, (j + 1) * w // gx
                p = cv2.goodFeaturesToTrack(gray[y0:y1, x0:x1], 150, 0.01, 8)
                if p is not None:
                    p = p.reshape(-1, 2) + np.array([x0, y0], np.float32)
                    pts.append(p)
        return np.vstack(pts).reshape(-1, 1, 2).astype(np.float32) if pts else None

    def step(self, gray):
        if self.prev is None:
            self.prev = gray
            return np.eye(3), True
        H, ok = np.eye(3), False
        p0 = self._features(self.prev)
        if p0 is not None and len(p0) >= 10:
            p1, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, p0, None, winSize=(21, 21), maxLevel=3)
            good = st.ravel() == 1
            if good.sum() >= 10:
                Hc, inl = cv2.findHomography(p0[good], p1[good], cv2.RANSAC, self.cfg.ransac_px)
                if Hc is not None and inl.sum() >= self.cfg.min_inliers:
                    H, ok = Hc, True
        self.prev = gray
        return H, ok

## 2. DETECTORES (intercambiables: cualquier callable que devuelva Nx6 [x1,y1,x2,y2,conf,cls])

In [4]:
class YoloDetector:
    """Solo DETECTA (predict), no hace tracking. El tracking lo hace nuestro Tracker."""

    def __init__(self, weights="yolo11n.pt", imgsz=1280, conf=0.10, classes=None,
                 device=None, tiles=None, overlap=0.2):
        from ultralytics import YOLO
        self.model = YOLO(weights)
        self.names = self.model.names
        self.imgsz, self.conf, self.classes, self.device = imgsz, conf, classes, device
        self.tiles, self.overlap = tiles, overlap      # tiles=(filas, columnas) o None

    def _predict(self, img):
        r = self.model.predict(img, imgsz=self.imgsz, conf=self.conf, classes=self.classes,
                               device=self.device, verbose=False)[0]
        b = getattr(r, "obb", None)          # modelos *-obb (DOTA): cajas orientadas; se usa su caja horizontal
        if b is None:
            b = r.boxes
        if b is None or len(b) == 0:
            return np.zeros((0, 6), np.float32)
        return np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None],
                               b.cls.cpu().numpy()[:, None]], 1).astype(np.float32)

    def __call__(self, frame_bgr, scale):
        if not self.tiles:
            d = self._predict(frame_bgr)
        else:
            import torch, torchvision
            H, W = frame_bgr.shape[:2]
            rows, cols = self.tiles
            th, tw = H / rows, W / cols
            outs = []
            for i in range(rows):
                for j in range(cols):
                    y0, y1 = int(max(0, i * th - self.overlap * th)), int(min(H, (i + 1) * th + self.overlap * th))
                    x0, x1 = int(max(0, j * tw - self.overlap * tw)), int(min(W, (j + 1) * tw + self.overlap * tw))
                    t = self._predict(frame_bgr[y0:y1, x0:x1])
                    t[:, [0, 2]] += x0
                    t[:, [1, 3]] += y0
                    outs.append(t)
            d = np.concatenate(outs, 0) if outs else np.zeros((0, 6), np.float32)
            if len(d):
                keep = torchvision.ops.batched_nms(torch.from_numpy(d[:, :4]), torch.from_numpy(d[:, 4]),
                                                   torch.from_numpy(d[:, 5]).long(), 0.5).numpy()
                d = d[keep]
        d[:, :4] *= scale          # a coordenadas de trabajo
        return d

## 3. PROPUESTAS DE MOVIMIENTO (agnósticas a la clase)

In [5]:
class MotionProposals:
    """Cambios respecto al frame t-gap, ya compensando la cámara. Sirven para objetos que YOLO no conoce."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        self.gap = max(2, int(round(cfg.proposal_gap_sec * fps)))
        self.grays = deque(maxlen=self.gap + 1)
        self.Hs = deque(maxlen=self.gap)
        self.oks = deque(maxlen=self.gap)
        self.last_diff = None

    def step(self, gray, H, ok):
        c = self.cfg
        if len(self.grays) > 0:
            self.Hs.append(H)
            self.oks.append(ok)
        self.grays.append(gray)
        if len(self.grays) < self.gap + 1 or not all(self.oks):
            return np.zeros((0, 6), np.float32)
        M = np.eye(3)
        for h in self.Hs:                       # H_t @ ... @ H_{t-gap+1}
            M = h @ M
        prev, curr = self.grays[0], gray
        h_, w_ = curr.shape
        warped = cv2.warpPerspective(prev, M, (w_, h_))
        valid = cv2.erode(cv2.warpPerspective(np.full((h_, w_), 255, np.uint8), M, (w_, h_)),
                          np.ones((11, 11), np.uint8))
        d = cv2.bitwise_and(cv2.absdiff(curr, warped), valid)
        self.last_diff = d
        if valid.any() and d[valid > 0].mean() > c.max_residual:
            return np.zeros((0, 6), np.float32)
        m = (d > c.diff_thr).astype(np.uint8) * 255
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
        m = cv2.dilate(m, np.ones((c.dilate, c.dilate), np.uint8))
        n, _, s, _ = cv2.connectedComponentsWithStats(m)
        gx = cv2.Sobel(curr, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(curr, cv2.CV_32F, 0, 1, ksize=3)
        out = []
        for k in range(1, n):
            x, y, bw, bh, a = s[k]
            if not (c.min_area <= a <= c.max_area):
                continue
            if x < c.border or y < c.border or x + bw > w_ - c.border or y + bh > h_ - c.border:
                continue
            ar = bw / max(bh, 1)
            if ar > c.max_aspect or ar < 1 / c.max_aspect:
                continue
            sx, sy = gx[y:y + bh, x:x + bw], gy[y:y + bh, x:x + bw]
            jxx, jyy, jxy = float((sx * sx).sum()), float((sy * sy).sum()), float((sx * sy).sum())
            coh = np.sqrt((jxx - jyy) ** 2 + 4 * jxy ** 2) / (jxx + jyy + 1e-6)
            if coh > c.max_coherence:
                continue                           # línea/cable/borde: el centroide del cambio resbala a lo largo (apertura)
            out.append([x, y, x + bw, y + bh, c.proposal_conf, -1])
        return np.array(out, np.float32).reshape(-1, 6)

## 4. TRACKER EN COORDENADAS ESTABILIZADAS (asociación estilo ByteTrack)

In [6]:
def iou_pad(a, b, pad):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = a + np.array([-pad, -pad, pad, pad]); b = b + np.array([-pad, -pad, pad, pad])
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)


class Track:
    def __init__(self, tid, det, fidx, maxlen):
        self.id = tid
        self.box = det[:4].astype(np.float32).copy()
        self.hits, self.miss = 1, 0
        self.votes = defaultdict(float)
        self.counts = defaultdict(int)
        self._vote(det)
        self.pts = np.array([self._center()], np.float32)   # historial de centros, siempre en coords del frame ACTUAL
        self.gs = np.zeros((1, 2), np.float32)             # flujo que la cámara induce en el track (H·c - c), por frame
        self._g = np.zeros(2, np.float32)
        self.par, self.par_frames = False, 0
        self.straight = 1.0
        self.toggles = 0
        self.maxlen = maxlen
        self.first, self.last = fidx, fidx
        self.disp, self.moving = 0.0, False
        self.vec = np.zeros(2, np.float32)
        self.on_cnt = self.off_cnt = 0
        self.moving_frames = 0
        self.max_disp = 0.0

    def _center(self):
        return [(self.box[0] + self.box[2]) / 2, (self.box[1] + self.box[3]) / 2]

    min_conf, min_hits = 0.40, 3                    # se fijan desde Config en Tracker.__init__

    def _vote(self, det):
        if det[5] >= 0 and det[4] >= Track.min_conf:
            self.votes[int(det[5])] += float(det[4]); self.counts[int(det[5])] += 1

    @property
    def cls(self):
        if not self.votes:
            return -1
        c = max(self.votes, key=self.votes.get)
        return c if self.counts[c] >= Track.min_hits else -1

    def warp(self, H):
        old_c = np.array(self._center(), np.float32)
        c = warp_pts(H, [self.box[:2], self.box[2:]])
        self.box = c.reshape(-1).astype(np.float32)
        self._g = (np.array(self._center(), np.float32) - old_c)        # flujo del suelo en este punto
        self.pts = warp_pts(H, self.pts)
        A = (H[:2, :2] / H[2, 2]).astype(np.float32)
        self.gs = self.gs @ A.T                                         # llevar el historial al frame actual

    def update(self, det, fidx):
        self.box = det[:4].astype(np.float32).copy()
        self.hits += 1; self.miss = 0; self.last = fidx
        self._vote(det)
        self.pts = np.vstack([self.pts, self._center()])[-self.maxlen:]
        self.gs = np.vstack([self.gs, self._g])[-self.maxlen:]


class Tracker:
    def __init__(self, cfg, fps):
        self.cfg = cfg
        Track.min_conf, Track.min_hits = cfg.cls_min_conf, cfg.cls_min_hits
        self.tracks, self.nid = [], 1
        self.max_miss = max(2, int(round(cfg.max_miss_sec * fps)))
        self.maxlen = int(round(cfg.window_sec * fps)) + 1

    @staticmethod
    def _compat(t, d):
        return t.cls < 0 or d[5] < 0 or t.cls == int(d[5])

    def _match(self, tracks, dets):
        if not tracks or len(dets) == 0:
            return [], list(range(len(tracks))), list(range(len(dets)))
        tb = np.array([t.box for t in tracks])
        iou = iou_pad(tb, dets[:, :4], self.cfg.box_pad)
        for i, t in enumerate(tracks):
            for j in range(len(dets)):
                if not self._compat(t, dets[j]):
                    iou[i, j] = 0
        r, c = linear_sum_assignment(1 - iou)
        pairs = [(i, j) for i, j in zip(r, c) if iou[i, j] >= self.cfg.match_iou]
        mi = {i for i, _ in pairs}; mj = {j for _, j in pairs}
        return pairs, [i for i in range(len(tracks)) if i not in mi], [j for j in range(len(dets)) if j not in mj]

    def update(self, dets, H, H_ok, fidx):
        c = self.cfg
        for t in self.tracks:                      # compensación de cámara: predicción de cada track
            t.warp(H)
            if not H_ok:                           # sin homografía fiable no hay historial fiable
                t.pts = t.pts[-1:]; t.gs = t.gs[-1:]
        high = dets[dets[:, 4] >= c.conf_high] if len(dets) else dets
        low = dets[(dets[:, 4] >= c.conf_low) & (dets[:, 4] < c.conf_high)] if len(dets) else dets

        pairs, ut, uh = self._match(self.tracks, high)
        for i, j in pairs:
            self.tracks[i].update(high[j], fidx)
        rem = [self.tracks[i] for i in ut]
        pairs2, ut2, _ = self._match(rem, low)     # 2ª asociación con detecciones débiles
        for i, j in pairs2:
            rem[i].update(low[j], fidx)
        matched_ids = {self.tracks[i].id for i, _ in pairs} | {rem[i].id for i, _ in pairs2}
        for t in self.tracks:
            if t.id not in matched_ids:
                t.miss += 1
        for j in uh:                               # tracks nuevos solo con detecciones fuertes
            self.tracks.append(Track(self.nid, high[j], fidx, self.maxlen)); self.nid += 1
        self.tracks = [t for t in self.tracks if t.miss <= self.max_miss]
        return [t for t in self.tracks if t.hits >= c.min_hits and t.miss == 0]

## 5. ESTADO DE MOVIMIENTO: desplazamiento NETO en coordenadas estabilizadas

In [7]:
class MotionState:
    def __init__(self, cfg, fps, skip_ids=()):
        self.cfg = cfg
        self.W = int(round(cfg.window_sec * fps))
        self.thr = cfg.min_move_px
        self.skip_ids = set(skip_ids)

    @staticmethod
    def _straightness(p):
        """Desplazamiento neto / recorrido, sobre 6 tramos (medianas) para no contar el jitter de la caja."""
        if len(p) < 6:
            return 1.0
        m = np.array([np.median(ch, 0) for ch in np.array_split(p, 6)])
        path = float(np.linalg.norm(np.diff(m, axis=0), axis=1).sum())
        return float(np.linalg.norm(m[-1] - m[0]) / (path + 1e-6))

    def _parallax_like(self, t, n):
        """Un objeto estático por encima del suelo deja un residuo PARALELO (y del mismo sentido) al flujo que la
        cámara induce en el suelo en ese punto, con razón h/Z = |residuo| / |flujo|. Si cumple, no es movimiento propio."""
        c = self.cfg
        g = t.gs[-n:]
        G = g.sum(0); Gn = float(np.linalg.norm(G))   # recorrido NETO del suelo (el ruido de H en hover no cuenta)
        if Gn < 1e-6 or t.disp < 1e-6:
            return False
        cos = float(t.vec @ G) / (t.disp * Gn)
        return cos >= np.cos(np.deg2rad(c.parallax_angle_deg)) and t.disp <= c.parallax_ratio * Gn

    def update(self, tracks, H_ok):
        c = self.cfg
        if not H_ok:
            return
        disps = []
        for t in tracks:
            n = min(len(t.pts), self.W + 1)
            if n < max(5, int(0.5 * self.W)):
                t.disp = 0.0; t.vec = np.zeros(2, np.float32); t.par = False; continue
            p = t.pts[-n:]; k = max(1, n // 5)
            t.vec = (np.median(p[-k:], 0) - np.median(p[:k], 0)).astype(np.float32)
            t.disp = float(np.linalg.norm(t.vec))
            disps.append(t.disp)
        self.thr = c.min_move_px
        if c.adaptive and len(disps) >= 5:         # solo útil si la mayoría de los tracks son estáticos
            med = np.median(disps); mad = np.median(np.abs(np.array(disps) - med)) * 1.4826
            self.thr = float(min(max(c.min_move_px, med + c.mad_k * mad), c.max_adaptive_px))
        for t in tracks:
            n = min(len(t.pts), self.W + 1)
            if len(t.pts) < max(5, int(0.5 * self.W)):
                continue
            t.straight = self._straightness(t.pts[-n:])
            size = float(np.sqrt(max((t.box[2] - t.box[0]) * (t.box[3] - t.box[1]), 1.0)))
            thr_t = max(self.thr, c.rel_move * size)   # un objeto grande necesita moverse más para contar
            raw = t.disp > thr_t and t.straight >= c.min_straightness
            t.par = False
            if raw and c.parallax_filter and t.cls not in self.skip_ids and self._parallax_like(t, n):
                raw, t.par = False, True               # explicable por paralaje: no cuenta como movimiento
                t.par_frames += 1
            if raw:
                t.on_cnt += 1; t.off_cnt = 0
            else:
                t.off_cnt += 1; t.on_cnt = 0
            if not t.moving and t.on_cnt >= c.on_frames:
                t.moving = True; t.toggles += 1
            elif t.moving and t.off_cnt >= c.off_frames:
                t.moving = False; t.toggles += 1
            t.moving_frames += int(t.moving)
            t.max_disp = max(t.max_disp, t.disp)

## 6. VISUALIZACIÓN

In [8]:
GREEN, RED, ORANGE = (60, 200, 60), (40, 40, 255), (0, 150, 255)


def fuse_moving(tracks, cfg):
    """Agrupa tracks en movimiento que están pegados y comparten velocidad (p. ej. las mitades de un camión)."""
    mv = [t for t in tracks if t.moving]
    parent = list(range(len(mv)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i

    cosmin = np.cos(np.deg2rad(cfg.merge_angle_deg))
    for i in range(len(mv)):
        for j in range(i + 1, len(mv)):
            a, b = mv[i], mv[j]
            va, vb = np.linalg.norm(a.vec), np.linalg.norm(b.vec)
            if va < 1e-6 or vb < 1e-6:
                continue
            if float(a.vec @ b.vec) / (va * vb) < cosmin or abs(va - vb) > cfg.merge_speed_ratio * max(va, vb):
                continue
            dx = max(0.0, max(a.box[0], b.box[0]) - min(a.box[2], b.box[2]))
            dy = max(0.0, max(a.box[1], b.box[1]) - min(a.box[3], b.box[3]))
            side = max(a.box[2] - a.box[0], a.box[3] - a.box[1], b.box[2] - b.box[0], b.box[3] - b.box[1])
            if np.hypot(dx, dy) <= cfg.merge_gap * side:
                parent[find(i)] = find(j)
    groups = defaultdict(list)
    for i, t in enumerate(mv):
        groups[find(i)].append(t)
    return list(groups.values())


def draw(frame, tracks, k, names, thr, cfg):
    for t in tracks:                                # estáticos: una caja por track
        if t.moving or t.cls < 0:
            continue
        x1, y1, x2, y2 = (t.box * k).astype(int)
        cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 2)
        cv2.putText(frame, f"#{t.id} {names.get(t.cls, '?')}", (x1, max(14, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, GREEN, 1, cv2.LINE_AA)
    if cfg.show_parallax:
        for t in tracks:
            if t.par and not t.moving:
                x1, y1, x2, y2 = (t.box * k).astype(int)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (170, 170, 170), 1)
                cv2.putText(frame, "paralaje?", (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (170, 170, 170), 1, cv2.LINE_AA)
    groups = fuse_moving(tracks, cfg) if cfg.merge_motion else [[t] for t in tracks if t.moving]
    for g in groups:
        box = np.array([min(t.box[0] for t in g), min(t.box[1] for t in g),
                        max(t.box[2] for t in g), max(t.box[3] for t in g)])
        known = [t for t in g if t.cls >= 0]
        main = max(known or g, key=lambda t: (t.box[2] - t.box[0]) * (t.box[3] - t.box[1]))
        lab = names.get(main.cls, "?") if known else "desconocido"
        color = RED if known else ORANGE
        x1, y1, x2, y2 = (box * k).astype(int)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"#{min(t.id for t in g)} {lab} {np.mean([t.disp for t in g]):.0f}px",
                    (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        if len(main.pts) > 1:
            cv2.polylines(frame, [(main.pts * k).astype(np.int32).reshape(-1, 1, 2)], False, color, 1)
    n_static = sum(1 for t in tracks if not t.moving and t.cls >= 0)
    cv2.putText(frame, f"En movimiento: {len(groups)} | Estaticos: {n_static} | umbral: {thr:.1f}px",
                (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

## 7. PIPELINE

In [9]:
def real_fps(video):
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps and 1 < fps <= 240:
        cap.release(); return fps
    n, last = 0, 0                                 # fps absurdo (p. ej. 1000 en algunos .webm): se mide con timestamps
    while cap.grab():
        n += 1; last = cap.get(cv2.CAP_PROP_POS_MSEC)
    cap.release()
    return n / (last / 1000.0) if last > 0 else 30.0


def run(video, detector=None, cfg=None, out="resultado_movimiento.mp4", summary_csv="tracks_resumen.csv",
        start=0, max_frames=None, names=None):
    cfg = cfg or Config()
    names = names or (getattr(detector, "names", None) or {})
    fps = real_fps(video)
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir {video}")
    W0, H0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = cfg.work_width / W0                    # original -> trabajo
    ww, wh = cfg.work_width, int(round(H0 * scale))
    ow = cfg.out_width or W0
    oh = int(round(H0 * ow / W0))
    k_out = ow / ww                                # trabajo -> salida
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    writer = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh))
    writer_dbg = None
    if cfg.debug_residual:
        writer_dbg = cv2.VideoWriter(str(out).replace(".mp4", "_residuo.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh))

    skip_ids = [i for i, n in names.items() if str(n).lower() in cfg.parallax_skip]
    gm, trk, ms = GlobalMotion(cfg), Tracker(cfg, fps), MotionState(cfg, fps, skip_ids)
    prop = MotionProposals(cfg, fps) if cfg.use_proposals else None
    history, i, bad = {}, 0, 0
    while True:
        ok, frame = cap.read()
        if not ok or (max_frames is not None and i >= max_frames):
            break
        small = cv2.resize(frame, (ww, wh), interpolation=cv2.INTER_AREA)
        gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (3, 3), 0)
        H, h_ok = gm.step(gray)
        bad += int(not h_ok)

        dets = detector(frame, scale) if detector is not None else np.zeros((0, 6), np.float32)
        if len(dets):                                # cajas gigantes = falsos positivos (casas, terrenos)
            area = (dets[:, 2] - dets[:, 0]) * (dets[:, 3] - dets[:, 1])
            dets = dets[area <= cfg.max_box_area_frac * ww * wh]
        if prop is not None:
            pr = prop.step(gray, H, h_ok)
            if len(pr) and len(dets):                # si YOLO ya explica el cambio, la propuesta sobra
                pr = pr[iou_pad(pr[:, :4], dets[:, :4], 2.0).max(1) < 0.1]
            if len(pr):
                dets = np.vstack([dets, pr]) if len(dets) else pr

        shown = trk.update(dets, H, h_ok, start + i)
        ms.update(shown, h_ok)
        for t in shown:
            history[t.id] = t
        vis = cv2.resize(frame, (ow, oh), interpolation=cv2.INTER_AREA)
        draw(vis, shown, k_out, names, ms.thr, cfg)
        writer.write(vis)
        if writer_dbg is not None and prop is not None and prop.last_diff is not None:
            res = cv2.cvtColor(cv2.convertScaleAbs(prop.last_diff, alpha=8), cv2.COLOR_GRAY2BGR)
            res = cv2.resize(res, (ow, oh))
            draw(res, shown, k_out, names, ms.thr, cfg)
            writer_dbg.write(res)
        i += 1
        if i % 100 == 0:
            print(f"{i} fotogramas")
    cap.release(); writer.release()
    if writer_dbg is not None:
        writer_dbg.release()

    rows = []
    for t in history.values():
        rows.append(dict(id=t.id, clase=names.get(t.cls, "desconocido") if t.cls >= 0 else "desconocido",
                         primer_frame=t.first, ultimo_frame=t.last, detecciones=t.hits,
                         frames_en_movimiento=t.moving_frames, frames_paralaje=t.par_frames, desplazamiento_max_px=round(t.max_disp, 1),
                         estado_final="movimiento" if t.moving else "estatico"))
    with open(summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["id"])
        w.writeheader(); w.writerows(rows)
    print(f"FPS={fps:.2f} | frames={i} | frames sin homografía fiable={bad} | tracks={len(rows)}")
    return rows

## 8. TRACKERS EXTERNOS (ByteTrack / BoT-SORT de Ultralytics) Y COMPARATIVA

In [10]:
class _Dets:
    """Imita lo mínimo del objeto Boxes de Ultralytics que consumen sus trackers (xyxy, xywh, conf, cls, indexado)."""

    def __init__(self, xyxy, conf, cls):
        self.xyxy = np.asarray(xyxy, np.float32).reshape(-1, 4)
        self.conf = np.asarray(conf, np.float32).reshape(-1)
        self.cls = np.asarray(cls, np.float32).reshape(-1)
        x1, y1, x2, y2 = self.xyxy.T if len(self.xyxy) else (np.zeros(0),) * 4
        self.xywh = np.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], 1).astype(np.float32) if len(self.xyxy) else np.zeros((0, 4), np.float32)
        self.data = np.concatenate([self.xyxy, self.conf[:, None], self.cls[:, None]], 1)

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, idx):
        return _Dets(self.xyxy[idx], self.conf[idx], self.cls[idx])


class UltralyticsTracker:
    """ByteTrack o BoT-SORT reales de Ultralytics, alimentados con NUESTRAS detecciones (así la comparativa es justa).
    kind: "bytetrack" | "botsort".  gmc_method (solo botsort): "sparseOptFlow" | "orb" | "ecc" | "none"."""

    FILES = {"bytetrack": "bytetrack.yaml", "botsort": "botsort.yaml"}

    def __init__(self, kind, cfg, fps, gmc_method=None, overrides=None):
        try:
            from ultralytics.utils import IterableSimpleNamespace, yaml_load
            from ultralytics.utils.checks import check_yaml
            from ultralytics.trackers.byte_tracker import BYTETracker
            from ultralytics.trackers.bot_sort import BOTSORT
        except Exception as e:
            raise ImportError("No pude importar los trackers de Ultralytics (¿versión distinta?). "
                              f"Ejecuta selftest_ultralytics() para ver el detalle. Error original: {e}")
        args = IterableSimpleNamespace(**yaml_load(check_yaml(self.FILES[kind])))
        args.track_high_thresh = cfg.conf_high
        args.track_low_thresh = cfg.conf_low
        args.new_track_thresh = cfg.conf_high
        args.track_buffer = int(round(cfg.max_miss_sec * 30))     # Ultralytics lo reescala con frame_rate/30
        if kind == "botsort":
            args.with_reid = False                                 # sin ReID: no requiere pesos adicionales
            if gmc_method is not None:
                args.gmc_method = gmc_method
        for k, v in (overrides or {}).items():
            setattr(args, k, v)
        self.kind, self.args = kind, args
        self.tracker = (BYTETracker if kind == "bytetrack" else BOTSORT)(args=args, frame_rate=int(round(fps)))

    def update(self, dets, img):
        """dets: Nx6 [x1,y1,x2,y2,conf,cls] en coordenadas de trabajo; img: frame BGR de trabajo (lo usa el GMC)."""
        d = _Dets(dets[:, :4], dets[:, 4], dets[:, 5]) if len(dets) else _Dets([], [], [])
        try:
            out = self.tracker.update(d, img)
        except TypeError:
            out = self.tracker.update(d)
        out = np.asarray(out, np.float32)
        return out[:, :7] if out.ndim == 2 and len(out) else np.zeros((0, 7), np.float32)   # x1,y1,x2,y2,id,conf,cls


def selftest_ultralytics(cfg=None, fps=30.0):
    """Ejecútala ANTES de la comparativa: comprueba que los trackers de Ultralytics responden como espera este código."""
    cfg = cfg or Config()
    img = (np.random.default_rng(0).random((540, 960, 3)) * 255).astype(np.uint8)
    d = np.array([[100, 100, 140, 160, 0.9, 2], [300, 200, 360, 260, 0.8, 0]], np.float32)
    for kind, gmc in (("bytetrack", None), ("botsort", "sparseOptFlow")):
        t = UltralyticsTracker(kind, cfg, fps, gmc_method=gmc)
        for _ in range(3):
            out = t.update(d, img)
        print(f"{kind:10s} OK | salida {out.shape} | ids {out[:, 4].astype(int).tolist()}")
        if kind == "botsort":
            try:
                w = t.tracker.gmc.apply(img, d[:, :4])
                print("           GMC OK | warp =", np.round(np.asarray(w), 3).tolist())
            except Exception as e:
                print("           GMC FALLA (Ultralytics lo ignora en silencio y no compensaría la cámara):", e)


class TrackBook:
    """Convierte la salida de un tracker externo (ids + cajas) en objetos Track propios, para reutilizar
    EXACTAMENTE la misma decisión de movimiento (MotionState) y el mismo dibujo en todos los trackers."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        Track.min_conf, Track.min_hits = cfg.cls_min_conf, cfg.cls_min_hits
        self.tracks = {}
        self.max_miss = max(2, int(round(cfg.max_miss_sec * fps)))
        self.maxlen = int(round(cfg.window_sec * fps)) + 1

    def update(self, out, H, H_ok, fidx):
        for t in self.tracks.values():
            t.warp(H)
            if not H_ok:
                t.pts = t.pts[-1:]; t.gs = t.gs[-1:]
        seen = set()
        for r in out:
            tid = int(r[4])
            det = np.array([r[0], r[1], r[2], r[3], r[5], r[6]], np.float32)
            if tid in self.tracks:
                self.tracks[tid].update(det, fidx)
            else:
                self.tracks[tid] = Track(tid, det, fidx, self.maxlen)
            seen.add(tid)
        for tid in list(self.tracks):
            if tid not in seen:
                self.tracks[tid].miss += 1
                if self.tracks[tid].miss > self.max_miss:
                    del self.tracks[tid]
        return [t for t in self.tracks.values() if t.hits >= self.cfg.min_hits and t.miss == 0]


def _rows(history, names):
    rows = []
    for t in history.values():
        rows.append(dict(id=t.id, clase=names.get(t.cls, "desconocido") if t.cls >= 0 else "desconocido",
                         primer_frame=t.first, ultimo_frame=t.last, detecciones=t.hits,
                         frames_en_movimiento=t.moving_frames, frames_paralaje=t.par_frames,
                         cambios_de_estado=t.toggles, desplazamiento_max_px=round(t.max_disp, 1),
                         estado_final="movimiento" if t.moving else "estatico"))
    return rows


def _metrics(kind, history, n_frames, fps, shown_counts):
    hs = list(history.values())
    if not hs:
        return dict(variante=kind, tracks=0)
    lens = np.array([t.hits for t in hs])
    minutos = max(n_frames / fps / 60.0, 1e-9)
    return dict(variante=kind,
                tracks=len(hs),
                ids_por_minuto=round(len(hs) / minutos, 1),
                duracion_mediana_s=round(float(np.median(lens)) / fps, 2),
                tracks_menores_1s_pct=round(100.0 * float(np.mean(lens < fps)), 1),
                objetos_simultaneos_prom=round(float(np.mean(shown_counts)), 2),
                pct_frames_en_movimiento=round(100.0 * sum(t.moving_frames for t in hs) / max(int(lens.sum()), 1), 1),
                cambios_estado_por_track=round(float(np.mean([t.toggles for t in hs])), 2))


def compare(video, detector=None, cfg=None, kinds=("propio", "bytetrack", "botsort"), out_dir="comparativa",
            start=0, max_frames=None, names=None, mosaic=True, extra=None, botsort_gmc="sparseOptFlow"):
    """Una sola pasada: homografía y detecciones se calculan UNA vez y se reparten a cada tracker.
    kinds: "propio", "bytetrack", "botsort", "botsort_sin_gmc" (o nombres de `extra`: {nombre: fábrica(cfg, fps)}
    donde la fábrica devuelve un objeto con .update(dets, img_bgr) -> Nx7 [x1,y1,x2,y2,id,conf,cls])."""
    cfg = cfg or Config()
    extra = extra or {}
    names = names or (getattr(detector, "names", None) or {})
    os.makedirs(out_dir, exist_ok=True)
    fps = real_fps(video)
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir {video}")
    W0, H0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = cfg.work_width / W0
    ww, wh = cfg.work_width, int(round(H0 * scale))
    ow = cfg.out_width or W0
    oh = int(round(H0 * ow / W0))
    k_out = ow / ww
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    skip_ids = [i for i, n in names.items() if str(n).lower() in cfg.parallax_skip]
    V = {}
    for kind in kinds:
        if kind == "propio":
            trk, ext, book = Tracker(cfg, fps), None, None
        else:
            if kind in extra:
                ext = extra[kind](cfg, fps)
            elif kind == "bytetrack":
                ext = UltralyticsTracker("bytetrack", cfg, fps)
            elif kind == "botsort":
                ext = UltralyticsTracker("botsort", cfg, fps, gmc_method=botsort_gmc)
            elif kind == "botsort_sin_gmc":
                ext = UltralyticsTracker("botsort", cfg, fps, gmc_method="none")
            else:
                raise ValueError(f"Variante desconocida: {kind}")
            trk, book = None, TrackBook(cfg, fps)
        V[kind] = dict(trk=trk, ext=ext, book=book, ms=MotionState(cfg, fps, skip_ids), hist={}, counts=[],
                       writer=cv2.VideoWriter(os.path.join(out_dir, f"{kind}.mp4"), fourcc, fps, (ow, oh)))
    pw = 960
    ph = int(round(oh * pw / ow))
    cols = 2
    rows_n = int(np.ceil(len(kinds) / cols))
    wmos = cv2.VideoWriter(os.path.join(out_dir, "mosaico.mp4"), fourcc, fps, (pw * cols, ph * rows_n)) if mosaic else None

    gm = GlobalMotion(cfg)
    prop = MotionProposals(cfg, fps) if cfg.use_proposals else None
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok or (max_frames is not None and i >= max_frames):
            break
        small = cv2.resize(frame, (ww, wh), interpolation=cv2.INTER_AREA)
        gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (3, 3), 0)
        H, h_ok = gm.step(gray)
        dets = detector(frame, scale) if detector is not None else np.zeros((0, 6), np.float32)
        if len(dets):
            area = (dets[:, 2] - dets[:, 0]) * (dets[:, 3] - dets[:, 1])
            dets = dets[area <= cfg.max_box_area_frac * ww * wh]
        if prop is not None:
            pr = prop.step(gray, H, h_ok)
            if len(pr) and len(dets):
                pr = pr[iou_pad(pr[:, :4], dets[:, :4], 2.0).max(1) < 0.1]
            if len(pr):
                dets = np.vstack([dets, pr]) if len(dets) else pr
        fidx = start + i
        vis0 = cv2.resize(frame, (ow, oh), interpolation=cv2.INTER_AREA)
        panels = []
        for kind, v in V.items():
            if v["trk"] is not None:
                shown = v["trk"].update(dets, H, h_ok, fidx)
            else:
                shown = v["book"].update(v["ext"].update(dets, small), H, h_ok, fidx)
            v["ms"].update(shown, h_ok)
            for t in shown:
                v["hist"][t.id] = t
            v["counts"].append(len(shown))
            vis = vis0.copy()
            draw(vis, shown, k_out, names, v["ms"].thr, cfg)
            cv2.putText(vis, kind.upper(), (20, oh - 25), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3, cv2.LINE_AA)
            v["writer"].write(vis)
            if wmos is not None:
                panels.append(cv2.resize(vis, (pw, ph), interpolation=cv2.INTER_AREA))
        if wmos is not None:
            while len(panels) < cols * rows_n:
                panels.append(np.zeros_like(panels[0]))
            wmos.write(np.vstack([np.hstack(panels[r * cols:(r + 1) * cols]) for r in range(rows_n)]))
        i += 1
        if i % 100 == 0:
            print(f"{i} fotogramas")
    cap.release()
    for v in V.values():
        v["writer"].release()
    if wmos is not None:
        wmos.release()

    metrics = []
    for kind, v in V.items():
        rows = _rows(v["hist"], names)
        with open(os.path.join(out_dir, f"tracks_{kind}.csv"), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["id"])
            w.writeheader(); w.writerows(rows)
        metrics.append(_metrics(kind, v["hist"], i, fps, v["counts"]))
    keys = []
    for m in metrics:
        keys += [k for k in m if k not in keys]
    with open(os.path.join(out_dir, "comparativa_metricas.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(metrics)
    print(f"FPS={fps:.2f} | frames={i}")
    return metrics

## Verificación de Ultralytics (ejecutar primero)

In [11]:
# Ejecuta esto PRIMERO: comprueba que los trackers de Ultralytics responden como espera este código
selftest_ultralytics(Config(), fps=30.0)

ImportError: No pude importar los trackers de Ultralytics (¿versión distinta?). Ejecuta selftest_ultralytics() para ver el detalle. Error original: cannot import name 'yaml_load' from 'ultralytics.utils' (/opt/anaconda3/lib/python3.13/site-packages/ultralytics/utils/__init__.py)

## Comparativa
Salidas en `comparativa/`: `mosaico.mp4` (todos lado a lado), un video por variante, `tracks_<variante>.csv` y `comparativa_metricas.csv`.

**Cómo leer las métricas** (sin verdad de terreno):
- `ids_por_minuto` y `tracks_menores_1s_pct` altos → el tracker **fragmenta** (pierde y reasigna IDs). Con la cámara en movimiento es el síntoma típico de no compensarla.
- `duracion_mediana_s` baja → lo mismo.
- `pct_frames_en_movimiento` y `cambios_estado_por_track` altos en una escena mayormente estática → **falsos movimientos**.
- Ninguna sustituye mirar el mosaico: un tracker puede tener pocas fragmentaciones y aun así asociar mal.

In [12]:
from pathlib import Path
import pandas as pd
from IPython.display import Video, display

VIDEO_PATH = Path("input1.mp4")     # o un clip corto
cfg = Config(work_width=960, out_width=1920)
detector = YoloDetector("yolo11n-obb.pt", imgsz=1280, conf=0.15)

metricas = compare(VIDEO_PATH, detector, cfg,
                   kinds=("propio", "bytetrack", "botsort"),   # añade "botsort_sin_gmc" para aislar el efecto del GMC
                   out_dir="comparativa", max_frames=600)
display(pd.DataFrame(metricas).set_index("variante"))

ImportError: No pude importar los trackers de Ultralytics (¿versión distinta?). Ejecuta selftest_ultralytics() para ver el detalle. Error original: cannot import name 'yaml_load' from 'ultralytics.utils' (/opt/anaconda3/lib/python3.13/site-packages/ultralytics/utils/__init__.py)

In [ ]:
display(Video("comparativa/mosaico.mp4", embed=False, width=1100))

## Siguiente paso recomendado
Si quieres números y no solo proxies, marca a mano 10-20 objetos estáticos conocidos (postes, coches aparcados) en un clip corto. Con eso se puede medir directamente cuántos de ellos acaba marcando como "en movimiento" cada tracker y si conservan un único ID.